# TE02_004 - Statusjoy Command Execution Validation

This notebook validates TE02_004 using the `/statusjoy` topic while the official remote controller is operated manually.

KPI requirement: precise execution of commands with 100% accuracy compared to the official remote controller.

## Validation Method

The official remote controller is used to command the real low-level system. The expected execution result is observed through `/statusjoy`.

The relevant `/statusjoy` states are:

| Value | Meaning |
|---:|---|
| `0` | No battery power |
| `1` | Battery power on, engine off |
| `128` | Engine on |

The test procedure is repeated 30 times. Depending on whether the action is engine start, engine stop, or battery cut-off, the `/statusjoy` value must transition between the valid states. For 30 repetitions, the KPI expects at least 60 valid state changes.

The KPI passes when:

- At least 60 valid `/statusjoy` transitions are detected.
- All observed stable states are one of `0`, `1`, or `128`.
- No invalid transition is detected.
- The measured transition success rate is 100%.

## Workflow

1. Run the configuration and logger cells.
2. Start the acquisition cell.
3. Operate the official remote controller for 30 repetitions.
4. Use the remote to produce transitions between `0`, `1`, and `128`.
5. Run the analysis cells after acquisition finishes.

If you are validating from a rosbag instead of the live machine, start the acquisition cell first and then play the bag in a terminal.

In [1]:
from pathlib import Path
import time
import os

import numpy as np
import pandas as pd
import rclpy
from rclpy.node import Node
from std_msgs.msg import UInt8

KPI_DIR = Path(str(os.environ.get('HOME')) + '/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004')
KPI_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV = KPI_DIR / 'te02_004_statusjoy_raw_samples.csv'
STABLE_CSV = KPI_DIR / 'te02_004_statusjoy_stable_states.csv'
TRANSITIONS_CSV = KPI_DIR / 'te02_004_statusjoy_transitions.csv'
SUMMARY_CSV = KPI_DIR / 'te02_004_statusjoy_summary.csv'

STATUSJOY_TOPIC = '/statusjoy'
VALID_STATES = {0: 'no_battery_power', 1: 'battery_on_engine_off', 128: 'engine_on'}
VALID_TRANSITIONS = {
    (0, 1): 'battery_power_on',
    (1, 0): 'battery_power_cut_off',
    (1, 128): 'engine_start',
    (128, 1): 'engine_stop',
    (128, 0): 'engine_stop_and_battery_cut_off',
    (0, 128): 'battery_power_on_and_engine_start',
}

EXPECTED_REPETITIONS = 15
MIN_REQUIRED_TRANSITIONS = 15
ACQUISITION_DURATION_S = 300.0
USE_SIM_TIME = False

# Debounce repeated publications of the same state. A new stable state is only accepted
# when it differs from the previous accepted state and persists for this long.
MIN_STABLE_DURATION_S = 0.05

print(f'Output directory: {KPI_DIR}')
print(f'Subscribing to: {STATUSJOY_TOPIC}')
print(f'Valid states: {VALID_STATES}')
print(f'Minimum required transitions: {MIN_REQUIRED_TRANSITIONS}')

Output directory: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_004
Subscribing to: /statusjoy
Valid states: {0: 'no_battery_power', 1: 'battery_on_engine_off', 128: 'engine_on'}
Minimum required transitions: 15


In [ ]:
class StatusjoyLogger(Node):
    def __init__(self):
        super().__init__('te02_004_statusjoy_logger')
        if USE_SIM_TIME:
            self.set_parameters([rclpy.parameter.Parameter('use_sim_time', rclpy.Parameter.Type.BOOL, True)])

        self.rows = []
        self.create_subscription(UInt8, STATUSJOY_TOPIC, self.statusjoy_cb, 10)
        self.create_timer(10.0, self.health_cb)

    def statusjoy_cb(self, msg):
        now_ns = self.get_clock().now().nanoseconds
        self.rows.append({
            'time_ns': now_ns,
            'time_s': now_ns / 1e9,
            'statusjoy': int(msg.data),
            'state_name': VALID_STATES.get(int(msg.data), 'unexpected_value'),
        })

    def health_cb(self):
        latest = self.rows[-1]['statusjoy'] if self.rows else None
        self.get_logger().info(f'/statusjoy samples={len(self.rows)}, latest={latest}')

In [ ]:
if not rclpy.ok():
    rclpy.init()

logger = StatusjoyLogger()
deadline = time.monotonic() + ACQUISITION_DURATION_S
print(f'Acquiring /statusjoy for {ACQUISITION_DURATION_S:.1f} s.')
print('Operate the official remote controller now. Press Ctrl+C here to stop earlier.')

try:
    while time.monotonic() < deadline:
        rclpy.spin_once(logger, timeout_sec=0.1)
except KeyboardInterrupt:
    print('Acquisition interrupted by user.')
finally:
    raw_df = pd.DataFrame(logger.rows)
    logger.destroy_node()

raw_df.to_csv(RAW_CSV, index=False)
print(f'Raw samples: {len(raw_df)}')
print(f'Wrote: {RAW_CSV}')
display(raw_df.head())
display(raw_df.tail())

In [ ]:
def extract_stable_states(raw_df: pd.DataFrame) -> pd.DataFrame:
    if raw_df.empty:
        return pd.DataFrame(columns=['start_s', 'end_s', 'duration_s', 'statusjoy', 'state_name', 'sample_count'])

    df = raw_df.sort_values('time_s').reset_index(drop=True).copy()
    groups = []
    start_idx = 0
    current_value = int(df.loc[0, 'statusjoy'])

    for idx in range(1, len(df)):
        value = int(df.loc[idx, 'statusjoy'])
        if value != current_value:
            end_idx = idx - 1
            start_s = float(df.loc[start_idx, 'time_s'])
            end_s = float(df.loc[end_idx, 'time_s'])
            duration_s = max(0.0, end_s - start_s)
            groups.append({
                'start_s': start_s,
                'end_s': end_s,
                'duration_s': duration_s,
                'statusjoy': current_value,
                'state_name': VALID_STATES.get(current_value, 'unexpected_value'),
                'sample_count': end_idx - start_idx + 1,
            })
            start_idx = idx
            current_value = value

    end_idx = len(df) - 1
    start_s = float(df.loc[start_idx, 'time_s'])
    end_s = float(df.loc[end_idx, 'time_s'])
    groups.append({
        'start_s': start_s,
        'end_s': end_s,
        'duration_s': max(0.0, end_s - start_s),
        'statusjoy': current_value,
        'state_name': VALID_STATES.get(current_value, 'unexpected_value'),
        'sample_count': end_idx - start_idx + 1,
    })

    stable = pd.DataFrame(groups)
    if stable.empty:
        return stable

    # Keep short segments if they are the only sample of a changed state, but mark stability explicitly.
    stable['stable_enough'] = (stable['duration_s'] >= MIN_STABLE_DURATION_S) | (stable['sample_count'] >= 2)
    return stable.reset_index(drop=True)

stable_df = extract_stable_states(raw_df)
stable_df.to_csv(STABLE_CSV, index=False)
print(f'Stable state segments: {len(stable_df)}')
print(f'Wrote: {STABLE_CSV}')
display(stable_df.head(20))

In [ ]:
def extract_transitions(stable_df: pd.DataFrame) -> pd.DataFrame:
    if stable_df.empty or len(stable_df) < 2:
        return pd.DataFrame(columns=[
            'transition_index', 'time_s', 'from_value', 'to_value', 'from_state', 'to_state',
            'transition', 'valid_state_values', 'valid_transition', 'status'
        ])

    accepted = stable_df[stable_df['stable_enough']].reset_index(drop=True)
    rows = []
    for idx in range(1, len(accepted)):
        from_value = int(accepted.loc[idx - 1, 'statusjoy'])
        to_value = int(accepted.loc[idx, 'statusjoy'])
        if from_value == to_value:
            continue

        valid_state_values = from_value in VALID_STATES and to_value in VALID_STATES
        transition_name = VALID_TRANSITIONS.get((from_value, to_value), 'unexpected_transition')
        valid_transition = valid_state_values and (from_value, to_value) in VALID_TRANSITIONS
        rows.append({
            'transition_index': len(rows) + 1,
            'time_s': float(accepted.loc[idx, 'start_s']),
            'from_value': from_value,
            'to_value': to_value,
            'from_state': VALID_STATES.get(from_value, 'unexpected_value'),
            'to_state': VALID_STATES.get(to_value, 'unexpected_value'),
            'transition': transition_name,
            'valid_state_values': valid_state_values,
            'valid_transition': valid_transition,
            'status': 'PASS' if valid_transition else 'FAIL',
        })
    return pd.DataFrame(rows)

transitions_df = extract_transitions(stable_df)
transitions_df.to_csv(TRANSITIONS_CSV, index=False)
print(f'Detected transitions: {len(transitions_df)}')
print(f'Wrote: {TRANSITIONS_CSV}')
display(transitions_df.head(80))

In [ ]:
observed_values = sorted(raw_df['statusjoy'].dropna().astype(int).unique().tolist()) if not raw_df.empty else []
unexpected_values = [value for value in observed_values if value not in VALID_STATES]
valid_transition_count = int(transitions_df['valid_transition'].sum()) if not transitions_df.empty else 0
invalid_transition_count = int((~transitions_df['valid_transition']).sum()) if not transitions_df.empty else 0
transition_count = int(len(transitions_df))
success_rate_pct = 100.0 * valid_transition_count / transition_count if transition_count else np.nan
if raw_df.empty:
    overall_status = 'INCONCLUSIVE'
    reason = 'No /statusjoy samples were received during acquisition.'
elif unexpected_values:
    overall_status = 'FAIL'
    reason = f'Unexpected /statusjoy values observed: {unexpected_values}.'
elif transition_count < MIN_REQUIRED_TRANSITIONS:
    overall_status = 'FAIL'
    reason = f'Only {transition_count} transitions detected; required at least {MIN_REQUIRED_TRANSITIONS}.'
elif invalid_transition_count > 0:
    overall_status = 'FAIL'
    reason = f'{invalid_transition_count} invalid transitions detected.'
elif success_rate_pct == 100.0:
    overall_status = 'PASS'
    reason = f'{valid_transition_count}/{transition_count} transitions are valid; success rate is 100%.'
else:
    overall_status = 'FAIL'
    reason = f'Transition success rate is {success_rate_pct:.2f}%, expected 100%.'

summary_rows = [
    {'metric': 'raw_statusjoy_samples', 'value': len(raw_df), 'status': 'INFO', 'reason': ''},
    {'metric': 'observed_values', 'value': str(observed_values), 'status': 'PASS' if not unexpected_values else 'FAIL', 'reason': f'unexpected={unexpected_values}'},
    {'metric': 'stable_state_segments', 'value': len(stable_df), 'status': 'INFO', 'reason': ''},
    {'metric': 'required_transitions', 'value': MIN_REQUIRED_TRANSITIONS, 'status': 'INFO', 'reason': f'{EXPECTED_REPETITIONS} repetitions require at least {MIN_REQUIRED_TRANSITIONS} state changes'},
    {'metric': 'detected_transitions', 'value': transition_count, 'status': 'PASS' if transition_count >= MIN_REQUIRED_TRANSITIONS else 'FAIL', 'reason': ''},
    {'metric': 'valid_transitions', 'value': valid_transition_count, 'status': 'PASS' if invalid_transition_count == 0 and transition_count > 0 else 'FAIL', 'reason': f'invalid={invalid_transition_count}'},
    {'metric': 'transition_success_rate_pct', 'value': success_rate_pct, 'status': 'PASS' if success_rate_pct == 100.0 else 'FAIL', 'reason': 'expected 100%'},
    {'metric': 'TE02_004_overall', 'value': overall_status, 'status': overall_status, 'reason': reason},
]

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df)
print(f'Wrote: {SUMMARY_CSV}')
print(f'Overall TE02_004 status: {overall_status}')
print(reason)

## Interpretation

Use `te02_004_statusjoy_summary.csv` as the KPI result table.

Use `te02_004_statusjoy_transitions.csv` as detailed evidence of each official remote-controller command execution. Each row is one detected state change in `/statusjoy`.

A `PASS` means the official remote-controller operation produced at least 60 valid state changes, all observed values were valid, all transitions were valid, and the transition success rate was 100%.